# SWNA Genomics Pipeline — Step-by-Step
**Package:** `whitenoise` · **Submodule:** `wn.genomics`

This notebook walks through the full pipeline for applying Stochastic White Noise Analysis (SWNA)
to a DNA sequence from a FASTA file. Each cell is one step. Run them in order the first time,
then jump to any individual step to re-run it independently.

**Framework:** Violanda et al. (2019), *Physica Scripta* 94, 125006

---
## Pipeline overview
```
FASTA  →  filter_sequence()  →  extract_pair()  →  compute_pair_msd() [save CSV]  →  fit_pair()  →  plot_pair()
                                                                                          ↑
                                                              refit_pair(msd_csv)  ────────
```

---
## Cell 1 — Imports
Run this cell once per session. Sets up the package path and UTF-8 encoding for Windows terminals.

In [ ]:
import sys, os

# Fix Unicode output on Windows
sys.stdout.reconfigure(encoding='utf-8')
sys.stderr.reconfigure(encoding='utf-8')

# Point to the whitenoise package (adjust if running from a different directory)
_REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
sys.path.insert(0, os.path.join(_REPO_ROOT, 'whitenoise'))

import numpy as np
import matplotlib
matplotlib.use('Agg')          # headless — change to 'TkAgg' or remove if you want interactive plots
import matplotlib.pyplot as plt

import whitenoise as wn
print('whitenoise version:', wn.__version__)
print('Available genomics functions:', [f for f in dir(wn.genomics) if not f.startswith('_')])

---
## Cell 2 — Configuration
**Edit these variables** to point to your FASTA file and choose which transition pair to analyze.
Everything downstream uses these variables — change them here and re-run the cells below.

In [ ]:
# ── Path to your FASTA file ────────────────────────────────────────────────────
FASTA_PATH = os.path.join(
    _REPO_ROOT,
    'whole genomes',
    'Conus textile isolate sp1 chromosome 1, whole genome shotgun sequence.fasta'
)

# ── Transition pair to analyze ─────────────────────────────────────────────────
# Same-type (diagonal): A->A, C->C, G->G, T->T  → best fit with DNA model
# Cross-type (off-diagonal): A->C, A->G, etc.   → try exponential or cosine model
FROM_NUC = 'A'
TO_NUC   = 'A'

# ── SWNA model ─────────────────────────────────────────────────────────────────
# 'dna'         → for diagonal pairs (same nucleotide)
# 'exponential' → for cross-type pairs or as an alternative
# 'cosine'      → for oscillatory behavior
# run wn.list_models() to see all options
MODEL = 'dna'

# ── MSD computation ────────────────────────────────────────────────────────────
# DNA model saturates at L ≈ 3/b. With b ≈ 0.025 (Conus chromosome), saturation
# is near L = 120. With the bacterial reference b ≈ 0.0024, saturation is near
# L = 1250. Use 1500 as a safe default that captures the full curve.
MAX_LAG = 1500

# ── Output directory ───────────────────────────────────────────────────────────
# MSD CSVs and plots will be saved here.
# Distances are NOT saved — see Cell 5 for the reason.
OUT_DIR = _REPO_ROOT

print(f'FASTA : {os.path.basename(FASTA_PATH)}')
print(f'Pair  : {FROM_NUC} -> {TO_NUC}')
print(f'Model : {MODEL}')
print(f'MaxLag: {MAX_LAG}')
print(f'OutDir: {OUT_DIR}')

---
## Cell 3 — Load and filter the sequence
Loads the first record from the FASTA and removes ambiguous IUPAC bases (W, S, N, R, Y, etc.).
Keep `seq` in memory — every step below reuses it without re-reading the file.

In [ ]:
record  = wn.genomics.read_fasta_single(FASTA_PATH)
seq_raw = record['sequence']
seq     = wn.genomics.filter_sequence(seq_raw)

print(f"ID          : {record['id']}")
print(f"Length (raw): {len(seq_raw):,} bp")
print(f"Ambiguous   : {record['n_ambiguous']:,} bases ({100*record['n_ambiguous']/len(seq_raw):.2f}%)")
print(f"Length (ACGT): {len(seq):,} bp")

---
## Cell 4 — (Optional) Summarize the full 4×4 transition matrix
Run this to see how many distances are available for all 16 pairs before committing to one.
Skip if you already know which pair you want.

In [ ]:
matrix = wn.genomics.get_all_transition_distances(seq)
wn.genomics.summarize_matrix(matrix)

---
## Cell 5 — Extract transition distances

> **Why we do NOT save distances to CSV for large genomes:**
> 
> A 179 Mbp chromosome produces ~50 million distances per pair. As a CSV that is ~350 MB
> per file (16 pairs = ~5.6 GB). Loading that CSV back into Python takes 15–30 seconds.
> Recomputing the distances directly from the in-memory sequence takes **under 8 seconds**.
> Saving and loading is slower than just recomputing.
> 
> Additionally, the distances array is not a scientific result — it is intermediate input
> to `compute_msd()`. The MSD array (Cell 6) is the meaningful checkpoint to save.
> 
> **Rule of thumb:** if `len(distances) > 1,000,000` — recompute, don't save.
> For small datasets (COI barcodes ~100–600 distances), saving is fine.

In [ ]:
import time

t0 = time.time()

# No save_path — see the note above
distances = wn.genomics.extract_pair(seq, FROM_NUC, TO_NUC)

print(f'\nExtracted in {time.time()-t0:.2f}s')
print(f'Min distance : {distances.min()}')
print(f'Max distance : {distances.max()}')
print(f'Mean distance: {distances.mean():.2f}')
print(f'Median       : {float(np.median(distances)):.2f}')

---
## Cell 6 — Compute empirical MSD
This is the step that takes the most time (~1–15 min depending on `MAX_LAG` and array size).
The MSD CSV is saved here — **this is the checkpoint worth keeping**.
Future sessions can skip Cells 3–6 and reload from the CSV directly in Cell 7.

In [ ]:
msd_csv = os.path.join(OUT_DIR, f'{FROM_NUC}_{TO_NUC}_msd.csv')

t0 = time.time()
lags, msd = wn.genomics.compute_pair_msd(distances, max_lag=MAX_LAG, save_path=msd_csv)
print(f'Computed in {time.time()-t0:.1f}s')

---
## Cell 6b — (Alternative) Reload MSD from a saved CSV
If you already ran Cell 6 in a previous session, run this cell instead of Cells 3–6.
Skips loading the FASTA and recomputing MSD entirely.

In [ ]:
# Uncomment and run this block to reload from a previously saved MSD CSV:

# msd_csv = os.path.join(OUT_DIR, f'{FROM_NUC}_{TO_NUC}_msd.csv')
# lags_raw, msd, _ = wn.read_csv(msd_csv)
# lags = lags_raw.astype(int)
# print(f'Reloaded {len(lags)} MSD points from {os.path.basename(msd_csv)}')

---
## Cell 7 — Fit a model
Fits the chosen SWNA model to the empirical MSD.

**Choosing `p0` (initial parameter guess):**
If the fit looks visually correct but R² is low, the optimizer found a local minimum.
Estimate `p0` from the MSD plot in Cell 8:
- `a` ≈ the plateau value (where the curve flattens)
- `c` ≈ `a - MSD[1]` (the initial gap below the plateau)
- `b` ≈ `6 / L_plateau` (where `L_plateau` is the lag where the curve visually flattens)

Then come back and re-run this cell with `P0 = [your_a, your_b, your_c]`.

In [ ]:
# Set to None to use defaults, or provide [a, b, c] for 'dna', [mu, beta] for 'exponential', etc.
P0     = None
BOUNDS = None     # e.g. ([0, 0, 0], [100, 1, 100]) for dna
MAX_LAG_FRACTION = 1.0   # use < 1.0 to exclude the noisy flat tail

fit = wn.genomics.fit_pair(
    (lags, msd),
    model=MODEL,
    p0=P0,
    bounds=BOUNDS,
    max_lag_fraction=MAX_LAG_FRACTION,
)

---
## Cell 8 — Plot the MSD fit
Saves the figure to a PNG. Change `save_path` to `None` if you only want to display inline.

In [ ]:
# Change the save path / title as needed
plot_title = f'{FROM_NUC}->{TO_NUC} | {MODEL} model'
plot_path  = os.path.join(OUT_DIR, f'{FROM_NUC}_{TO_NUC}_fit.png')

fig = wn.genomics.plot_pair(lags, msd, fit, title=plot_title, save_path=plot_path)

# Display inline in Jupyter
plt.show()

---
## Cell 9 — Re-fit with a different model or custom p0
Use this cell to try a different model or initial guess **without recomputing the MSD**.
The MSD CSV saved in Cell 6 is reloaded directly.

In [ ]:
# ── Change these to experiment ─────────────────────────────────────────────────
REFIT_MODEL = 'dna'             # try 'exponential', 'cosine', etc.
REFIT_P0    = [27.0, 0.025, 2.0]  # [a, b, c] for dna — set to None for defaults
REFIT_FRAC  = 1.0               # fraction of lags to use
REFIT_PATH  = os.path.join(OUT_DIR, f'{FROM_NUC}_{TO_NUC}_refit.png')
# ──────────────────────────────────────────────────────────────────────────────

fit_new, fig_new = wn.genomics.refit_pair(
    msd_csv,
    model=REFIT_MODEL,
    p0=REFIT_P0,
    max_lag_fraction=REFIT_FRAC,
    save_path=REFIT_PATH,
)

plt.show()

---
## Cell 10 — (Alternative) Full pipeline in one call
Runs all steps above in a single function call. Useful once you know the settings work.
Requires `seq` to be loaded (Cell 3). Set `save_dir` to save all outputs automatically.

In [ ]:
result = wn.genomics.analyze_pair(
    source   = seq,             # pass already-loaded sequence — skips re-reading FASTA
    from_nuc = FROM_NUC,
    to_nuc   = TO_NUC,
    model    = MODEL,
    max_lag  = MAX_LAG,
    save_dir = OUT_DIR,         # saves MSD CSV and plot PNG; set to None to skip saving
    p0       = None,
)

print('\nResult keys:', list(result.keys()))
print('n_distances :', result['n_distances'])
if result['fit']:
    print('R²          :', round(result['fit'].r_squared, 4))

plt.show()

---
## Cell 11 — Loop over multiple pairs
Runs the pipeline for a set of pairs. Loads the sequence once, then iterates.
Edit `PAIRS` and `MODELS` to match what you want to analyze.

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
# Diagonal pairs use 'dna'; cross-type pairs often fit better with 'exponential'
PAIRS  = [('A', 'A'), ('C', 'C'), ('G', 'G'), ('T', 'T')]
MODELS = {'same': 'dna', 'cross': 'exponential'}
# ─────────────────────────────────────────────────────────────────────────────

summary = []

for from_n, to_n in PAIRS:
    model = MODELS['same'] if from_n == to_n else MODELS['cross']
    result = wn.genomics.analyze_pair(
        source   = seq,
        from_nuc = from_n,
        to_nuc   = to_n,
        model    = model,
        max_lag  = MAX_LAG,
        save_dir = OUT_DIR,
        verbose  = False,
    )
    r2 = result['fit'].r_squared if result['fit'] else None
    summary.append({'pair': f'{from_n}->{to_n}', 'model': model,
                    'n': result['n_distances'], 'r2': r2})
    print(f"  {from_n}->{to_n}  R²={r2:.4f}" if r2 else f"  {from_n}->{to_n}  fit failed")

import pandas as pd
df = pd.DataFrame(summary)
print('\n', df.to_string(index=False))

---
## Notes

### On saving distances vs MSD
- **Distances** (50M rows, ~350 MB): recompute from FASTA in <8s. **Do not save.**
- **MSD** (1500 rows, ~50 KB): save this — it is the scientific checkpoint.

### On model choice
| Pair type | Recommended model | Notes |
|-----------|------------------|-------|
| Diagonal (A→A, etc.) | `dna` | Derived from same-type distances in Violanda et al. |
| Off-diagonal (A→C, etc.) | `exponential` or `cosine` | DNA model not derived for these |

### On poor fits (R² < 0.5)
1. Plot the MSD first (Cell 8) to see the shape visually.
2. Estimate `p0` from the plot and re-run Cell 9.
3. Try `max_lag_fraction=0.7` to exclude the noisy plateau tail.
4. Try a different model entirely.

### Reference
Violanda R R, Bernido C C, Carpio-Bernido M V (2019).  
*White noise functional integral for exponentially decaying memory: nucleotide distribution in bacterial genomes.*  
Physica Scripta 94, 125006.